<a id="top"></a>

# Demo: three outcomes, including a hallucinated call

```yaml
title: "Demo: three outcomes, including a hallucinated call"
keywords: three outcomes, hallucinated tool call, validation, application validates, tool_choice, haiku contrast, non-determinism, teacher demo
estimated duration: 13
```

> **Lesson:** L07. Demo 4 in the [roadmap](../../../../docs/origin/lesson_roadmaps/L07/demos_or_activities.md). Makes **live** Claude calls — set `ANTHROPIC_API_KEY` before running. Model behavior **varies**: if Sonnet is too well-behaved on P3, re-run on **Haiku 4.5**, or hand-edit the arguments (last cell) to force the validation path. Clear outputs before committing.
>
> **The client is LangChain's `ChatAnthropic`** (from L03). `bind_tools` makes `calculator` a tool; the model's request is `AIMessage.tool_calls`, and *your* code validates each call before running it. The API key still loads through the config seam (`require_anthropic_key`); never hard-coded.
>
> The point to land: handing the model a tool has **three observable outcomes** — calls it, skips it, or calls it with bad arguments — and the **application**, not the model, catches the bad case. The schema is a contract about shape, not a guarantee about behavior.

## Contents

- [1. Setup — the tool, plus a visible validator](#1-setup--the-tool-plus-a-visible-validator)
- [2. Outcome one — a clean tool call (P1)](#2-outcome-one--a-clean-tool-call-p1)
- [3. Outcome two — the model answers without the tool (P2)](#3-outcome-two--the-model-answers-without-the-tool-p2)
- [4. Outcome three — a malformed / hallucinated call (P3)](#4-outcome-three--a-malformed--hallucinated-call-p3)
- [5. Fallback — force the validation path by hand](#5-fallback--force-the-validation-path-by-hand)
- [6. Takeaways](#6-takeaways)

## 1. Setup — the tool, plus a visible validator

The same `calculator`, plus a `dispatch` helper whose validation step is **visible**: a bad tool call is *rejected*, not crashed-on. Two tool-bound models ready — Sonnet 4.6 (anchor) and Haiku 4.5 (the smaller-model contrast).

In [ ]:
from typing import Annotated, Any

from langchain_anthropic import ChatAnthropic
from langchain_core.messages import AIMessage, HumanMessage

from fluffy_potato_curriculum.common.config import require_anthropic_key

SONNET = "claude-sonnet-4-6"  # course anchor
HAIKU = "claude-haiku-4-5"  # smaller model -- fumbles tool selection / args more often


# The ONE tool that carries all four demos: a deterministic calculator.
# L07 is deliberately single-tool, single-round-trip (multi-tool is L08, the
# agent loop is L10). Resist adding a second tool.
def calculator(
    expression: Annotated[str, "The arithmetic expression to evaluate, e.g. '18374 * 92431'."],
) -> str:
    """Evaluate a basic arithmetic expression (the four operators and parentheses) and
    return the exact numeric result. Use this whenever the user asks for a calculation.

    Restricted to digits and the operators + - * / ( ) . and whitespace so a
    hallucinated expression cannot run arbitrary code. Raises ValueError on
    anything else -- that rejection is the whole point of Demo 4."""
    allowed = set("0123456789+-*/(). ")
    if not expression or set(expression) - allowed:
        raise ValueError(f"refusing to evaluate non-arithmetic expression: {expression!r}")
    # eval is safe here ONLY because the character whitelist above blocks names,
    # attributes, and calls. Never eval untrusted input without such a guard.
    return str(eval(expression))


def dispatch(call: dict[str, Any]) -> str:
    """Validate one tool call (a {name, args, id} dict) and run it, or return a loud rejection."""
    if call["name"] != "calculator":
        return f"REJECTED: no such tool {call['name']!r} (the model invented it)"
    args = call["args"]
    if "expression" not in args:
        return f"REJECTED: missing required argument 'expression' (got {args!r})"
    try:
        return f"OK: calculator({args['expression']!r}) = {calculator(args['expression'])}"
    except ValueError as exc:
        return f"REJECTED: {exc}"


# One tool-bound model per client. bind_tools carries the calculator definition on
# every call; the model only ever proposes a call, never runs the function.
MODELS: dict[str, Any] = {
    SONNET: ChatAnthropic(model=SONNET, api_key=require_anthropic_key(), max_tokens=400).bind_tools(
        [calculator]
    ),
    HAIKU: ChatAnthropic(model=HAIKU, api_key=require_anthropic_key(), max_tokens=400).bind_tools(
        [calculator]
    ),
}


def ask(model_name: str, prompt: str) -> AIMessage:
    """Invoke the named tool-bound model on one prompt and return its AIMessage."""
    return MODELS[model_name].invoke([HumanMessage(prompt)])


print("setup ready")

[↑ Back to top](#top)

## 2. Outcome one — a clean tool call (P1)

Arithmetic the model should defer to the tool. A tool call in `.tool_calls`, valid args, a real result.

In [ ]:
resp = ask(SONNET, "What is 47,219 multiplied by 8,803?")
print("tool_calls:", [c["name"] for c in resp.tool_calls], "| any text:", bool(resp.content))
for call in resp.tool_calls:
    print("dispatch ->", dispatch(call))

[↑ Back to top](#top)

## 3. Outcome two — the model answers without the tool (P2)

A trivial sum the model already owns. Usually it answers directly — text only, an empty `.tool_calls`. A registered tool is an **option**, never an obligation (adding a tool to a task the model already owns is wasted overhead — a forward-link to [L08](L0702_lecture.md)).

In [ ]:
resp = ask(SONNET, "What is 2 + 2?")
print("tool_calls:", [c["name"] for c in resp.tool_calls])
print("text:", resp.content)
print("did the model call the tool?", bool(resp.tool_calls))

[↑ Back to top](#top)

## 4. Outcome three — a malformed / hallucinated call (P3)

A prompt that nudges the model toward a bad argument (a non-numeric token in the expression). The model may emit a tool call whose `args` are malformed — and the **application's validation catches it**. If Sonnet sanitizes cleanly, the Haiku run usually fumbles.

In [ ]:
P3 = "Use the calculator to work out the average of these three: 12, 19, and twenty."

for model_name in (SONNET, HAIKU):
    resp = ask(model_name, P3)
    print(f"=== {model_name} ===")
    print("  tool_calls:", [c["name"] for c in resp.tool_calls])
    for call in resp.tool_calls:
        print("  proposed args:", call["args"])
        print("  dispatch ->", dispatch(call))

[↑ Back to top](#top)

## 5. Fallback — force the validation path by hand

If neither model fumbles on the day, hand-build a tool call with a bad value and feed it to `dispatch`. What matters is the application's **response** to a bad call — you can see it regardless of whether the model cooperated.

In [ ]:
# Hand-built tool calls (the LangChain {name, args, id} shape) mimicking three hallucinations.
bad_calls: list[dict[str, Any]] = [
    {"name": "calculator", "args": {"expression": "twenty + 1"}, "id": "b1", "type": "tool_call"},
    {"name": "calculator", "args": {}, "id": "b2", "type": "tool_call"},  # missing arg
    {"name": "wikipedia", "args": {"query": "geese"}, "id": "b3", "type": "tool_call"},  # invented
]
for call in bad_calls:
    print(dispatch(call))

[↑ Back to top](#top)

## 6. Takeaways

- The model can **hallucinate** a tool call: wrong types, missing required args, invented args, even a tool name that doesn't exist. The schema stopped **none** of it at generation time — **the application validates; the model proposes.**
- One hallucinated call will teach you more than ten clean runs — the L07 analogue of L06's tag-violation moment.
- Tool calls are **not deterministic**: same prompt + schema can call, skip, or pass different arguments across runs. That is *why* validation is not optional.
- `tool_choice` (auto / any / none / specific) can *bias* the decision, but forcing a call still does not guarantee well-formed arguments. Designing the error *response* is [L08](L0702_lecture.md)'s job, not L07's.

[↑ Back to top](#top)